# Model Context Protocol (MCP)

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/model-context-protocol)

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

This notebook builds a **tiny MCP-style server and client from scratch** so you can watch the
`tools/list` → `tools/call` JSON-RPC exchange happen, then shows the same thing with the
official `mcp` SDK.

## 1. The message format: JSON-RPC 2.0

Every MCP message is a JSON-RPC request (`method`, `params`, `id`) or a reply (same `id`, plus
`result` or `error`). The `id` is what pairs a reply with the request that caused it.

In [ ]:
import json

def request(method, params=None, id=1):
    msg = {"jsonrpc": "2.0", "id": id, "method": method}
    if params is not None:
        msg["params"] = params
    return msg

print(json.dumps(request("tools/list"), indent=2))

## 2. A minimal server

A server is just a registry of tools plus a dispatcher that answers `tools/list` and `tools/call`.
Each tool advertises a JSON-Schema `inputSchema` — the same function-calling contract used everywhere else.

In [ ]:
WEATHER = {"Paris": "14\u00b0C, light rain", "Cairo": "33\u00b0C, sunny"}

TOOLS = {
    "get_weather": {
        "description": "Current weather for a city.",
        "inputSchema": {"type": "object",
                        "properties": {"city": {"type": "string"}},
                        "required": ["city"]},
        "fn": lambda city: f"{city}: {WEATHER.get(city, 'unknown')}",
    }
}

def server(req):
    """Dispatch one JSON-RPC request, return the JSON-RPC reply."""
    rid, method = req["id"], req["method"]
    if method == "tools/list":
        listed = [{"name": n, "description": t["description"],
                   "inputSchema": t["inputSchema"]} for n, t in TOOLS.items()]
        return {"jsonrpc": "2.0", "id": rid, "result": {"tools": listed}}
    if method == "tools/call":
        p = req["params"]
        out = TOOLS[p["name"]]["fn"](**p["arguments"])
        return {"jsonrpc": "2.0", "id": rid,
                "result": {"content": [{"type": "text", "text": out}]}}
    return {"jsonrpc": "2.0", "id": rid,
            "error": {"code": -32601, "message": f"unknown method {method}"}}

## 3. The client: discover, then call

The client sends `tools/list` to learn what exists, then `tools/call` to invoke one. Watch the `id` fields pair up.

In [ ]:
# 1-2. discover
reply = server(request("tools/list", id=1))
print("available:", [t["name"] for t in reply["result"]["tools"]])

# 3-4. call
reply = server(request("tools/call",
                       {"name": "get_weather", "arguments": {"city": "Paris"}}, id=2))
print("result  :", reply["result"]["content"][0]["text"])

## 4. The same thing with the official SDK

In production you never hand-write JSON-RPC — the `mcp` package generates the schema from your
function's type hints and docstring. This cell shows the real server; run it as a standalone
script (`python weather_server.py`), not inside the notebook, since it takes over stdio.

```bash
pip install "mcp[cli]"
```

In [ ]:
server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("weather")

@mcp.tool()
def get_weather(city: str) -> str:
    """Current weather for a city."""
    fake = {"Paris": "14C, light rain", "Cairo": "33C, sunny"}
    return f"{city}: {fake.get(city, 'unknown')}"

if __name__ == "__main__":
    mcp.run()
'''
with open("weather_server.py", "w") as f:
    f.write(server_code)
print("wrote weather_server.py — run: python weather_server.py")

## ✏️ Your turn

Add a **second tool** `get_forecast(city, days)` to the from-scratch server, then call it.

- Give it an `inputSchema` with `city` (string) and `days` (integer), both required.
- Return a short text string.
- Verify with the assert.

In [ ]:
# TODO(you): register a 'get_forecast' tool in TOOLS
# TOOLS["get_forecast"] = { ... }

reply = server(request("tools/call",
               {"name": "get_forecast", "arguments": {"city": "Paris", "days": 3}}, id=3))
assert reply["result"]["content"][0]["text"]  # non-empty text result
print("ok:", reply["result"]["content"][0]["text"])

<details>
<summary>Solution</summary>

```python
TOOLS["get_forecast"] = {
    "description": "Multi-day forecast for a city.",
    "inputSchema": {"type": "object",
                    "properties": {"city": {"type": "string"},
                                   "days": {"type": "integer"}},
                    "required": ["city", "days"]},
    "fn": lambda city, days: f"{city}: {days}-day outlook — mild",
}
```
</details>